# BIRLA INSTITUTE OF TECHNOLOGY AND SCIENCE, PILANI
## Work Integrated Learning Programmes Division
### Deep Reinforcement Learning — Lab Assignment 1
#### Part 1: Multi-Armed Bandit (MAB)
#### Adaptive Treatment Recommendation System

| | |
|---|---|
| **Group Number** | 218 |
| **Submission Deadline** | 8th June, 2026 |
| **File** | Team 218 - MAB |


## Environment Info
_Timestamp and Virtual Machine ID (required for submission)_

In [ ]:
import platform, datetime, socket, os

print("=" * 60)
print("VIRTUAL LAB EXECUTION DETAILS")
print("=" * 60)
print(f"Timestamp        : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Hostname / VM ID : {socket.gethostname()}")
print(f"OS               : {platform.system()} {platform.release()}")
print(f"Python Version   : {platform.python_version()}")
print(f"Processor        : {platform.processor() or 'N/A'}")
print("=" * 60)


## Imports & Global Setup

In [ ]:
# Standard library and third-party imports
import numpy as np          # Numerical operations
import random               # Python random (for seed)
import pandas as pd         # DataFrame operations
import matplotlib.pyplot as plt  # Plotting
import math                 # For log() in UCB1

print("All libraries imported successfully.")


---
## Task 1: Dataset Design (1 Mark)

**Objective:** Generate the synthetic patient-treatment environment for Group 218.

### Parameter Derivation
- **Group Number:** G = 218
- **Number of Medicines:** K = (G mod 3) + 5
- **Hidden Success Probability:** P_i = 0.4 + ((G + i) mod 6) × 0.07
- **Seeds:** `random.seed(G)`, `numpy.random.seed(G)`


In [ ]:
# ── Set seeds for full reproducibility ──────────────────────────────────────
G = 218
random.seed(G)
np.random.seed(G)

# ── Compute number of medicines ──────────────────────────────────────────────
# Formula: K = (G mod 3) + 5
K = (G % 3) + 5

# ── Compute hidden success probabilities ─────────────────────────────────────
# Formula: P_i = 0.4 + ((G + i) mod 6) * 0.07  for i in {0, 1, ..., K-1}
true_probs = [0.4 + ((G + i) % 6) * 0.07 for i in range(K)]

# ── Display group parameters ─────────────────────────────────────────────────
print("=" * 60)
print("GROUP PARAMETERS")
print("=" * 60)
print(f"Group Number          : G = {G}")
print(f"Number of Medicines   : K = ({G} mod 3) + 5 = {G % 3} + 5 = {K}")
print()
print("Hidden Success Probabilities:")
for i, p in enumerate(true_probs):
    print(f"  Medicine {i}: P_{i} = 0.4 + (({G}+{i}) mod 6) * 0.07 = {p:.2f}")


In [ ]:
# ── Create base dataset with 1000 patient records ────────────────────────────
# patient_id    : 0 to 999
# severity_score: (patient_id mod 5) + 1  →  values 1 (mild) to 5 (critical)
# Other columns are filled dynamically by each algorithm

NUM_PATIENTS = 1000

def create_base_dataset(num_patients=1000):
    """
    Creates the synthetic patient dataset.

    Parameters
    ----------
    num_patients : int - total number of patient records (default 1000)

    Returns
    -------
    pd.DataFrame with columns: patient_id, severity_score,
    assigned_medicine (None), clinical_outcome (None), utility_score (None)
    """
    data = {
        'patient_id'        : list(range(num_patients)),
        'severity_score'    : [(pid % 5) + 1 for pid in range(num_patients)],
        'assigned_medicine' : [None] * num_patients,
        'clinical_outcome'  : [None] * num_patients,
        'utility_score'     : [None] * num_patients,
    }
    return pd.DataFrame(data)

base_df = create_base_dataset(NUM_PATIENTS)

print("First 10 rows of the base dataset:")
print(base_df.head(10).to_string(index=False))
print(f"\nDataset shape: {base_df.shape}")
print(f"Severity distribution: {base_df['severity_score'].value_counts().sort_index().to_dict()}")


---
### Helper Function: Simulate Patient Treatment

This function simulates the clinical outcome when a medicine is assigned to a patient.
It is called inside every strategy algorithm.


In [ ]:
def treat_patient(medicine_idx, severity):
    """
    Simulates the clinical treatment of one patient.

    Uses the hidden success probability of the selected medicine to draw a
    binary clinical outcome (Bernoulli trial), then computes the utility score
    adjusted for patient severity.

    Parameters
    ----------
    medicine_idx : int   - index of the chosen medicine (0 to K-1)
    severity     : int   - disease severity score (1=mild, 5=critical)

    Returns
    -------
    clinical_outcome : int   - 1 if recovered, 0 if not recovered
    utility_score    : float - outcome * (1 - severity/10)
                               e.g. severity=1, recovered → reward = 0.9
                                    severity=5, recovered → reward = 0.5
                                    not recovered         → reward = 0.0
    """
    p = true_probs[medicine_idx]                     # hidden success probability
    clinical_outcome = int(np.random.random() < p)   # Bernoulli draw
    utility_score    = clinical_outcome * (1 - severity / 10.0)
    return clinical_outcome, utility_score

# Quick sanity check
print("Sanity check for treat_patient():")
print(f"  Medicine 3 (P=0.75), severity=1 → expected avg utility ≈ {0.75*(1-1/10):.3f}")
outcomes = [treat_patient(3, 1)[1] for _ in range(5000)]
print(f"  Simulated avg utility over 5000 trials: {np.mean(outcomes):.3f}")


---
## Task 2: Immediate Exploitation Strategy — Greedy (1 Mark)

**Policy:** Test each medicine exactly 10 times (warm-up), then always prescribe
the medicine with the highest observed average utility.

- Warm-up phase: K × 10 = 70 patients → each medicine tried 10×
- Exploitation phase: remaining 930 patients → always pick argmax(avg_utility)


### Visual — Greedy strategy
_Two-phase flow: warm-up (all 7 medicines × 10 trials) then lock-in on the best arm._

In [ ]:
from IPython.display import display, HTML
display(HTML('''

<div style="background:#f8f9fc;border:1px solid #d0d8e8;border-radius:12px;padding:16px;margin:12px 0">
<p style="font-size:13px;font-weight:500;color:#2a3460;margin:0 0 8px 0">Illustration — Greedy (Immediate Exploitation) Strategy</p>
<svg width="100%" viewBox="0 0 680 390" xmlns="http://www.w3.org/2000/svg">
<defs><marker id="arr1" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M2 1L8 5L2 9" fill="none" stroke="context-stroke" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/></marker></defs>
<rect x="30" y="20" width="280" height="320" rx="12" fill="#eef2f8" stroke="#b8c4d8" stroke-width="1.2"/>
<text font-family="sans-serif" font-size="13" font-weight="500" fill="#2a3460" x="170" y="46" text-anchor="middle">Warm-up phase</text>
<text font-family="sans-serif" font-size="11" fill="#6a7490" x="170" y="63" text-anchor="middle">Each medicine tried 10 times (70 patients)</text>
<g transform="translate(50,88)"><rect width="28" height="66" rx="5" fill="#4682b4" stroke="#2a5a8a" stroke-width="1"/><rect x="7" y="0" width="14" height="11" rx="2" fill="#6aa0cc"/><rect x="9" y="68" width="10" height="7" rx="1" fill="#2a5a8a"/><text font-family="sans-serif" font-size="10" fill="white" x="14" y="38" text-anchor="middle">M0</text><text font-family="sans-serif" font-size="10" fill="#3a4460" x="14" y="87" text-anchor="middle">x10</text></g>
<g transform="translate(88,88)"><rect width="28" height="66" rx="5" fill="#e87320" stroke="#b85010" stroke-width="1"/><rect x="7" y="0" width="14" height="11" rx="2" fill="#f0a060"/><rect x="9" y="68" width="10" height="7" rx="1" fill="#b85010"/><text font-family="sans-serif" font-size="10" fill="white" x="14" y="38" text-anchor="middle">M1</text><text font-family="sans-serif" font-size="10" fill="#3a4460" x="14" y="87" text-anchor="middle">x10</text></g>
<g transform="translate(126,88)"><rect width="28" height="66" rx="5" fill="#2e8b57" stroke="#1a6040" stroke-width="1"/><rect x="7" y="0" width="14" height="11" rx="2" fill="#60b880"/><rect x="9" y="68" width="10" height="7" rx="1" fill="#1a6040"/><text font-family="sans-serif" font-size="10" fill="white" x="14" y="38" text-anchor="middle">M2</text><text font-family="sans-serif" font-size="10" fill="#3a4460" x="14" y="87" text-anchor="middle">x10</text></g>
<g transform="translate(164,88)"><rect width="28" height="66" rx="5" fill="#d4a800" stroke="#a07800" stroke-width="1.5"/><rect x="7" y="0" width="14" height="11" rx="2" fill="#f8d040"/><rect x="9" y="68" width="10" height="7" rx="1" fill="#a07800"/><text font-family="sans-serif" font-size="10" fill="#5a3800" x="14" y="38" text-anchor="middle">M3</text><text font-family="sans-serif" font-size="10" fill="#3a4460" x="14" y="87" text-anchor="middle">x10</text></g>
<g transform="translate(202,88)"><rect width="28" height="66" rx="5" fill="#c0392b" stroke="#8a1a10" stroke-width="1"/><rect x="7" y="0" width="14" height="11" rx="2" fill="#e07060"/><rect x="9" y="68" width="10" height="7" rx="1" fill="#8a1a10"/><text font-family="sans-serif" font-size="10" fill="white" x="14" y="38" text-anchor="middle">M4</text><text font-family="sans-serif" font-size="10" fill="#3a4460" x="14" y="87" text-anchor="middle">x10</text></g>
<g transform="translate(50,206)"><rect width="28" height="66" rx="5" fill="#8a2be2" stroke="#5a0ab0" stroke-width="1"/><rect x="7" y="0" width="14" height="11" rx="2" fill="#b070e8"/><rect x="9" y="68" width="10" height="7" rx="1" fill="#5a0ab0"/><text font-family="sans-serif" font-size="10" fill="white" x="14" y="38" text-anchor="middle">M5</text><text font-family="sans-serif" font-size="10" fill="#3a4460" x="14" y="87" text-anchor="middle">x10</text></g>
<g transform="translate(88,206)"><rect width="28" height="66" rx="5" fill="#008b8b" stroke="#006060" stroke-width="1"/><rect x="7" y="0" width="14" height="11" rx="2" fill="#40b8b8"/><rect x="9" y="68" width="10" height="7" rx="1" fill="#006060"/><text font-family="sans-serif" font-size="10" fill="white" x="14" y="38" text-anchor="middle">M6</text><text font-family="sans-serif" font-size="10" fill="#3a4460" x="14" y="87" text-anchor="middle">x10</text></g>
<rect x="148" y="208" width="138" height="46" rx="8" fill="#fff8e0" stroke="#c8a800" stroke-width="1.2"/>
<text font-family="sans-serif" font-size="11" font-weight="500" fill="#5a3800" x="217" y="227" text-anchor="middle">Compare averages</text>
<text font-family="sans-serif" font-size="10" fill="#7a5800" x="217" y="244" text-anchor="middle">argmax(avg_utility)</text>
<rect x="60" y="296" width="220" height="34" rx="6" fill="#f8f4e0" stroke="#c8c0a0" stroke-width="1"/>
<text font-family="sans-serif" font-size="10" fill="#5a3800" x="170" y="311" text-anchor="middle">Warm-up: patients 1-70</text>
<text font-family="sans-serif" font-size="10" fill="#5a3800" x="170" y="325" text-anchor="middle">round-robin across all 7 medicines</text>
<line x1="312" y1="180" x2="365" y2="180" stroke="#c0392b" stroke-width="2" marker-end="url(#arr1)"/>
<text font-family="sans-serif" font-size="11" font-weight="500" fill="#c0392b" x="338" y="173" text-anchor="middle">lock in!</text>
<rect x="368" y="20" width="282" height="320" rx="12" fill="#fffbe8" stroke="#c8a800" stroke-width="1.5"/>
<text font-family="sans-serif" font-size="13" font-weight="500" fill="#5a3800" x="509" y="46" text-anchor="middle">Exploitation phase</text>
<text font-family="sans-serif" font-size="11" fill="#7a5800" x="509" y="63" text-anchor="middle">Best medicine only — 930 patients</text>
<g transform="translate(472,96)"><rect width="52" height="108" rx="8" fill="#d4a800" stroke="#a07800" stroke-width="2"/><rect x="14" y="0" width="24" height="16" rx="3" fill="#f8d040"/><rect x="18" y="124" width="16" height="9" rx="2" fill="#a07800"/><text font-family="sans-serif" font-size="13" font-weight="500" fill="#5a3800" x="26" y="62" text-anchor="middle">M3</text><text font-family="sans-serif" font-size="10" fill="#7a5800" x="26" y="80" text-anchor="middle">P=0.75</text></g>
<text font-family="sans-serif" font-size="22" fill="#c8a800" x="540" y="130" text-anchor="middle">&#9733;</text>
<text font-family="sans-serif" font-size="11" fill="#7a5800" x="509" y="236" text-anchor="middle">Best arm — Medicine 3</text>
<rect x="388" y="256" width="244" height="58" rx="8" fill="#fff8e0" stroke="#c8a800" stroke-width="1"/>
<text font-family="sans-serif" font-size="10" fill="#7a5800" x="510" y="274" text-anchor="middle">Patients 71 to 1000</text>
<text font-family="sans-serif" font-size="10" fill="#7a5800" x="510" y="290" text-anchor="middle">always assigned M3</text>
<text font-family="sans-serif" font-size="10" fill="#7a5800" x="510" y="306" text-anchor="middle">smooth linear reward growth</text>
<text font-family="sans-serif" font-size="11" font-weight="500" fill="#505870" x="340" y="368" text-anchor="middle">Greedy — simple but risks early commitment to wrong medicine</text>
</svg></div>
'''))

Illustration — Greedy (Immediate Exploitation) Strategy 
 
 
 
 Warm-up phase 
 Each medicine tried 10 times (70 patients) 
 M0 x10 
 M1 x10 
 M2 x10 
 M3 x10 
 M4 x10 
 M5 x10 
 M6 x10 
 
 Compare averages 
 argmax(avg_utility) 
 
 Warm-up: patients 1-70 
 round-robin across all 7 medicines 
 
 lock in! 
 
 Exploitation phase 
 Best medicine only — 930 patients 
 M3 P=0.75 
 ★ 
 Best arm — Medicine 3 
 
 Patients 71 to 1000 
 always assigned M3 
 smooth linear reward growth 
 Greedy — simple but risks early commitment to wrong medicine

In [ ]:
def greedy_strategy(base_df, K, warm_up=10):
    """
    Immediate Exploitation (Greedy) Strategy.

    Phase 1 – Warm-up:
        Assign medicines in round-robin order (med = pid % K).
        Each medicine is tried exactly `warm_up` times before exploitation begins.

    Phase 2 – Exploitation:
        Always select the medicine with the highest average utility score so far.

    Parameters
    ----------
    base_df  : pd.DataFrame - base patient dataset
    K        : int          - number of medicines
    warm_up  : int          - trials per medicine before exploitation (default 10)

    Returns
    -------
    df               : pd.DataFrame  - dataset with filled columns
    cumulative_reward: list[float]   - cumulative utility after each patient
    """
    np.random.seed(G)   # reset seed for fair comparison across strategies
    df = base_df.copy()

    counts     = np.zeros(K)   # number of times each medicine was selected
    total_util = np.zeros(K)   # sum of utility scores per medicine
    avg_util   = np.zeros(K)   # average utility per medicine

    cumulative_reward = []
    cum_reward = 0.0

    for pid in range(len(df)):
        severity = df.loc[pid, 'severity_score']

        # ── Warm-up phase: round-robin assignment ─────────────────────────────
        if pid < warm_up * K:
            med = pid % K
        # ── Exploitation phase: best medicine by average utility ──────────────
        else:
            med = int(np.argmax(avg_util))

        # Simulate treatment outcome
        outcome, utility = treat_patient(med, severity)

        # Populate dataset columns
        df.loc[pid, 'assigned_medicine'] = med
        df.loc[pid, 'clinical_outcome']  = outcome
        df.loc[pid, 'utility_score']     = utility

        # Update running statistics
        counts[med]     += 1
        total_util[med] += utility
        avg_util[med]    = total_util[med] / counts[med]

        cum_reward += utility
        cumulative_reward.append(cum_reward)

    print("GREEDY STRATEGY RESULTS")
    print("=" * 50)
    print(f"Final Cumulative Reward : {cum_reward:.4f}")
    print(f"Best Medicine Found     : Medicine {np.argmax(avg_util)}")
    print(f"Medicine Selection Counts: {counts.astype(int)}")
    print(f"Average Utilities       : {np.round(avg_util, 4)}")
    return df, cumulative_reward

# Run the greedy strategy
greedy_df, greedy_rewards = greedy_strategy(base_df, K, warm_up=10)

print("\nFirst 10 rows after Greedy execution:")
print(greedy_df.head(10).to_string(index=False))


---
## Task 3: Controlled Clinical Trial Strategy — ε-Greedy (1.5 Marks)

**Policy:** With probability ε explore (random medicine); with probability 1-ε exploit
(best medicine so far). Main run uses ε = 10%, with analysis for ε = 1% and ε = 50%.


In [ ]:
def epsilon_greedy_strategy(base_df, K, epsilon=0.10, label=""):
    """
    Epsilon-Greedy Strategy for adaptive treatment recommendation.

    At each patient:
      - With probability epsilon     → explore: pick a random medicine
      - With probability 1 - epsilon → exploit: pick the best medicine so far

    This models the clinical principle of "mostly best treatment, occasionally
    try alternatives to discover hidden opportunities."

    Parameters
    ----------
    base_df  : pd.DataFrame - base patient dataset
    K        : int          - number of medicines
    epsilon  : float        - exploration probability (0.0 to 1.0)
    label    : str          - display label for output

    Returns
    -------
    df               : pd.DataFrame
    cumulative_reward: list[float]
    """
    np.random.seed(G)
    df = base_df.copy()

    counts     = np.zeros(K)
    total_util = np.zeros(K)
    avg_util   = np.zeros(K)

    cumulative_reward = []
    cum_reward    = 0.0
    explore_count = 0
    exploit_count = 0

    for pid in range(len(df)):
        severity = df.loc[pid, 'severity_score']

        # ── Epsilon-greedy medicine selection ─────────────────────────────────
        if counts.sum() == 0 or np.random.random() < epsilon:
            med = np.random.randint(0, K)   # explore: random arm
            explore_count += 1
        else:
            med = int(np.argmax(avg_util))  # exploit: best known arm
            exploit_count += 1

        outcome, utility = treat_patient(med, severity)

        df.loc[pid, 'assigned_medicine'] = med
        df.loc[pid, 'clinical_outcome']  = outcome
        df.loc[pid, 'utility_score']     = utility

        counts[med]     += 1
        total_util[med] += utility
        avg_util[med]    = total_util[med] / counts[med]

        cum_reward += utility
        cumulative_reward.append(cum_reward)

    tag = label if label else f"epsilon={epsilon}"
    print(f"EPSILON-GREEDY [{tag}]")
    print("=" * 50)
    print(f"Final Cumulative Reward : {cum_reward:.4f}")
    print(f"Explore / Exploit steps : {explore_count} / {exploit_count}")
    print(f"Best Medicine Found     : Medicine {np.argmax(avg_util)}")
    print(f"Medicine Selection Counts: {counts.astype(int)}")
    print(f"Average Utilities       : {np.round(avg_util, 4)}")
    print()
    return df, cumulative_reward


In [ ]:
# ── Main run: 10% exploration ─────────────────────────────────────────────────
eps_df_10, eps_rewards_10 = epsilon_greedy_strategy(base_df, K, epsilon=0.10,
                                                    label="10% Exploration (main)")
print("First 10 rows after ε-Greedy (10%) execution:")
print(eps_df_10.head(10).to_string(index=False))


In [ ]:
# ── Analysis: ε = 1% (very low exploration) ───────────────────────────────────
eps_df_01, eps_rewards_01 = epsilon_greedy_strategy(base_df, K, epsilon=0.01,
                                                    label="1% Exploration")
print("Observation: With only 1% exploration, the agent may lock onto a suboptimal")
print("medicine early if unlucky in the first few rounds.")


In [ ]:
# ── Analysis: ε = 50% (excessive exploration) ────────────────────────────────
eps_df_50, eps_rewards_50 = epsilon_greedy_strategy(base_df, K, epsilon=0.50,
                                                    label="50% Exploration")
print("Observation: 50% exploration wastes roughly half the patients on random")
print("medicines even late in the trial when the best arm is already known.")


### Analysis of Exploration Rates

| ε | Behaviour | Risk |
|---|---|---|
| **1%** | Almost pure exploitation after initial trials | May commit to a suboptimal medicine if early samples were unlucky |
| **10%** | Balanced — mostly exploits, occasionally explores | Good trade-off for 1000 patients |
| **50%** | Half of all patients receive random treatment | Excessive; cumulative reward is significantly lower |


### Visual — Epsilon-Greedy strategy
_Decision tree per patient + bar chart comparing ε = 1%, 10%, 50%._

In [ ]:
from IPython.display import display, HTML
display(HTML('''

<div style="background:#f8f9fc;border:1px solid #d0d8e8;border-radius:12px;padding:16px;margin:12px 0">
<p style="font-size:13px;font-weight:500;color:#2a3460;margin:0 0 8px 0">Illustration — Epsilon-Greedy (Controlled Clinical Trial) Strategy</p>
<svg width="100%" viewBox="0 0 680 410" xmlns="http://www.w3.org/2000/svg">
<defs><marker id="arr2" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M2 1L8 5L2 9" fill="none" stroke="context-stroke" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/></marker></defs>
<rect x="30" y="20" width="310" height="250" rx="12" fill="#f0f4fb" stroke="#b0bcd8" stroke-width="1"/>
<text font-family="sans-serif" font-size="13" font-weight="500" fill="#2a3460" x="185" y="44" text-anchor="middle">Per-patient decision</text>
<circle cx="185" cy="108" r="34" fill="#f8f4e8" stroke="#c8a800" stroke-width="2"/>
<text font-family="sans-serif" font-size="18" fill="#a07800" x="185" y="102" text-anchor="middle">?</text>
<text font-family="sans-serif" font-size="11" font-weight="500" fill="#7a5800" x="185" y="119" text-anchor="middle">coin flip</text>
<text font-family="sans-serif" font-size="10" fill="#7a5800" x="185" y="133" text-anchor="middle">random()</text>
<path d="M155 128 L90 176" fill="none" stroke="#c0392b" stroke-width="1.5" marker-end="url(#arr2)"/>
<text font-family="sans-serif" font-size="11" fill="#c0392b" x="102" y="156" text-anchor="middle">prob e</text>
<path d="M215 128 L280 176" fill="none" stroke="#2e8b57" stroke-width="1.5" marker-end="url(#arr2)"/>
<text font-family="sans-serif" font-size="11" fill="#2e8b57" x="270" y="156" text-anchor="middle">prob 1-e</text>
<rect x="46" y="176" width="92" height="52" rx="8" fill="#ffe8e8" stroke="#c0392b" stroke-width="1.2"/>
<text font-family="sans-serif" font-size="11" font-weight="500" fill="#8a1a10" x="92" y="197" text-anchor="middle">Explore</text>
<text font-family="sans-serif" font-size="10" fill="#8a1a10" x="92" y="212" text-anchor="middle">random medicine</text>
<text font-family="sans-serif" font-size="10" fill="#8a1a10" x="92" y="225" text-anchor="middle">M0-M6 equally</text>
<rect x="200" y="176" width="112" height="52" rx="8" fill="#e8f8ee" stroke="#2e8b57" stroke-width="1.2"/>
<text font-family="sans-serif" font-size="11" font-weight="500" fill="#1a6040" x="256" y="197" text-anchor="middle">Exploit</text>
<text font-family="sans-serif" font-size="10" fill="#1a6040" x="256" y="212" text-anchor="middle">best known arm</text>
<text font-family="sans-serif" font-size="10" fill="#1a6040" x="256" y="225" text-anchor="middle">argmax(avg_util)</text>
<rect x="75" y="244" width="220" height="14" rx="4" fill="#e0e4f0"/>
<text font-family="sans-serif" font-size="10" fill="#505870" x="185" y="256" text-anchor="middle">update avg_utility after each patient</text>
<rect x="358" y="20" width="292" height="250" rx="12" fill="#f8f6f0" stroke="#c8c0a0" stroke-width="1"/>
<text font-family="sans-serif" font-size="13" font-weight="500" fill="#2a3460" x="504" y="44" text-anchor="middle">Exploration rate comparison</text>
<text font-family="sans-serif" font-size="10" fill="#6a7490" x="504" y="60" text-anchor="middle">final cumulative reward (1000 patients)</text>
<rect x="378" y="88" width="52" height="162" rx="4" fill="#e07070" stroke="#c04040" stroke-width="1"/>
<text font-family="sans-serif" font-size="11" font-weight="500" fill="#7a2020" x="404" y="264" text-anchor="middle">403</text>
<text font-family="sans-serif" font-size="10" fill="#7a2020" x="404" y="278" text-anchor="middle">e=1%</text>
<rect x="454" y="56" width="52" height="194" rx="4" fill="#2e8b57" stroke="#1a6040" stroke-width="2"/>
<text font-family="sans-serif" font-size="11" font-weight="500" fill="#0a4020" x="480" y="264" text-anchor="middle">498</text>
<text font-family="sans-serif" font-size="10" fill="#0a4020" x="480" y="278" text-anchor="middle">e=10%</text>
<text font-family="sans-serif" font-size="14" fill="#2e8b57" x="492" y="70" text-anchor="middle">&#9733;</text>
<rect x="530" y="80" width="52" height="170" rx="4" fill="#e87320" stroke="#b85010" stroke-width="1"/>
<text font-family="sans-serif" font-size="11" font-weight="500" fill="#7a3010" x="556" y="264" text-anchor="middle">460</text>
<text font-family="sans-serif" font-size="10" fill="#7a3010" x="556" y="278" text-anchor="middle">e=50%</text>
<line x1="372" y1="250" x2="594" y2="250" stroke="#b0b8c0" stroke-width="0.8"/>
<rect x="30" y="284" width="620" height="70" rx="8" fill="#f0f4f8" stroke="#c0c8d8" stroke-width="1"/>
<text font-family="sans-serif" font-size="11" fill="#3a4460" x="340" y="306" text-anchor="middle">e=1%  - commits too early, risks suboptimal arm</text>
<text font-family="sans-serif" font-size="11" fill="#3a4460" x="340" y="324" text-anchor="middle">e=10% - best balance, discovers M3 reliably</text>
<text font-family="sans-serif" font-size="11" fill="#3a4460" x="340" y="342" text-anchor="middle">e=50% - too much waste on random trials late in simulation</text>
</svg></div>
'''))

Illustration — Epsilon-Greedy (Controlled Clinical Trial) Strategy 
 
 
 
 Per-patient decision 
 
 ? 
 coin flip 
 random() 
 
 prob e 
 
 prob 1-e 
 
 Explore 
 random medicine 
 M0-M6 equally 
 
 Exploit 
 best known arm 
 argmax(avg_util) 
 
 update avg_utility after each patient 
 
 Exploration rate comparison 
 final cumulative reward (1000 patients) 
 
 403 
 e=1% 
 
 498 
 e=10% 
 ★ 
 
 460 
 e=50% 
 
 
 e=1% - commits too early, risks suboptimal arm 
 e=10% - best balance, discovers M3 reliably 
 e=50% - too much waste on random trials late in simulation

---
## Task 4: Confidence-Based Strategy — UCB1 (1 Mark)

**Policy:** Select the medicine with the highest Upper Confidence Bound:

$$UCB_i(t) = \bar{u}_i + \sqrt{\frac{2 \ln t}{n_i}}$$

- $\bar{u}_i$ = average utility of medicine $i$
- $n_i$ = number of times medicine $i$ has been selected
- $t$ = current time step

Medicines with fewer observations get a large exploration bonus that **shrinks
automatically** as evidence accumulates — exactly as the senior physician recommends.


### Visual — UCB1 strategy
_Score bars for all 7 medicines showing avg utility (solid) + shrinking exploration bonus (hatched)._

In [ ]:
from IPython.display import display, HTML
display(HTML('''

<div style="background:#f8f9fc;border:1px solid #d0d8e8;border-radius:12px;padding:16px;margin:12px 0">
<p style="font-size:13px;font-weight:500;color:#2a3460;margin:0 0 8px 0">Illustration — UCB1 (Confidence-Based) Strategy</p>
<svg width="100%" viewBox="0 0 680 400" xmlns="http://www.w3.org/2000/svg">
<defs>
  <pattern id="hatch3" width="5" height="5" patternUnits="userSpaceOnUse" patternTransform="rotate(45)"><line x1="0" y1="0" x2="0" y2="5" stroke="#aaa" stroke-width="1.2"/></pattern>
</defs>
<text font-family="sans-serif" font-size="13" font-weight="500" fill="#2a3460" x="340" y="24" text-anchor="middle">UCB1 score = avg_utility + exploration bonus</text>
<text font-family="sans-serif" font-size="11" fill="#6a7490" x="340" y="42" text-anchor="middle">Bonus = sqrt(2 x ln(t) / n_i)  shrinks as count n_i grows</text>
<text font-family="sans-serif" font-size="11" fill="#3a4460" x="50" y="68" text-anchor="start">Medicine</text>
<text font-family="sans-serif" font-size="11" fill="#3a4460" x="118" y="68" text-anchor="start">UCB score (avg utility + bonus)</text>
<rect x="30" y="72" width="640" height="1.5" rx="1" fill="#d0d4e0"/>
<g transform="translate(0,0)">
  <text font-family="sans-serif" font-size="12" fill="#3a4460" x="50" y="102" text-anchor="start">M0</text>
  <rect x="115" y="86" width="108" height="22" rx="3" fill="#4682b4"/><rect x="223" y="86" width="62" height="22" rx="3" fill="url(#hatch3)" stroke="#4682b4" stroke-width="0.5"/>
  <text font-family="sans-serif" font-size="10" fill="white" x="169" y="101" text-anchor="middle">0.43</text><text font-family="sans-serif" font-size="10" fill="#4a5a6a" x="254" y="101" text-anchor="middle">+0.62</text>
</g>
<g transform="translate(0,30)">
  <text font-family="sans-serif" font-size="12" fill="#3a4460" x="50" y="102" text-anchor="start">M1</text>
  <rect x="115" y="86" width="122" height="22" rx="3" fill="#e87320"/><rect x="237" y="86" width="58" height="22" rx="3" fill="url(#hatch3)" stroke="#e87320" stroke-width="0.5"/>
  <text font-family="sans-serif" font-size="10" fill="white" x="176" y="101" text-anchor="middle">0.43</text><text font-family="sans-serif" font-size="10" fill="#4a5a6a" x="266" y="101" text-anchor="middle">+0.58</text>
</g>
<g transform="translate(0,60)">
  <text font-family="sans-serif" font-size="12" fill="#3a4460" x="50" y="102" text-anchor="start">M2</text>
  <rect x="115" y="86" width="138" height="22" rx="3" fill="#2e8b57"/><rect x="253" y="86" width="52" height="22" rx="3" fill="url(#hatch3)" stroke="#2e8b57" stroke-width="0.5"/>
  <text font-family="sans-serif" font-size="10" fill="white" x="184" y="101" text-anchor="middle">0.46</text><text font-family="sans-serif" font-size="10" fill="#4a5a6a" x="279" y="101" text-anchor="middle">+0.52</text>
</g>
<g transform="translate(0,90)">
  <text font-family="sans-serif" font-size="12" fill="#5a3800" x="50" y="102" text-anchor="start">M3</text>
  <rect x="115" y="82" width="218" height="26" rx="4" fill="#d4a800" stroke="#a07800" stroke-width="1.5"/><rect x="333" y="82" width="34" height="26" rx="4" fill="url(#hatch3)" stroke="#a07800" stroke-width="1"/>
  <text font-family="sans-serif" font-size="11" font-weight="500" fill="#3a2800" x="224" y="99" text-anchor="middle">0.54</text><text font-family="sans-serif" font-size="10" fill="#4a5a6a" x="350" y="99" text-anchor="middle">+0.34</text>
  <text font-family="sans-serif" font-size="12" fill="#c8a000" x="398" y="100" text-anchor="start">&#9733; selected 337/1000</text>
</g>
<g transform="translate(0,126)">
  <text font-family="sans-serif" font-size="12" fill="#3a4460" x="50" y="102" text-anchor="start">M4</text>
  <rect x="115" y="86" width="80" height="22" rx="3" fill="#c0392b"/><rect x="195" y="86" width="76" height="22" rx="3" fill="url(#hatch3)" stroke="#c0392b" stroke-width="0.5"/>
  <text font-family="sans-serif" font-size="10" fill="white" x="155" y="101" text-anchor="middle">0.30</text><text font-family="sans-serif" font-size="10" fill="#4a5a6a" x="233" y="101" text-anchor="middle">+0.76</text>
</g>
<g transform="translate(0,156)">
  <text font-family="sans-serif" font-size="12" fill="#3a4460" x="50" y="102" text-anchor="start">M5</text>
  <rect x="115" y="86" width="96" height="22" rx="3" fill="#8a2be2"/><rect x="211" y="86" width="72" height="22" rx="3" fill="url(#hatch3)" stroke="#8a2be2" stroke-width="0.5"/>
  <text font-family="sans-serif" font-size="10" fill="white" x="163" y="101" text-anchor="middle">0.36</text><text font-family="sans-serif" font-size="10" fill="#4a5a6a" x="247" y="101" text-anchor="middle">+0.72</text>
</g>
<g transform="translate(0,186)">
  <text font-family="sans-serif" font-size="12" fill="#3a4460" x="50" y="102" text-anchor="start">M6</text>
  <rect x="115" y="86" width="104" height="22" rx="3" fill="#008b8b"/><rect x="219" y="86" width="70" height="22" rx="3" fill="url(#hatch3)" stroke="#008b8b" stroke-width="0.5"/>
  <text font-family="sans-serif" font-size="10" fill="white" x="167" y="101" text-anchor="middle">0.37</text><text font-family="sans-serif" font-size="10" fill="#4a5a6a" x="254" y="101" text-anchor="middle">+0.70</text>
</g>
<rect x="30" y="316" width="300" height="22" rx="4" fill="#eef2f8" stroke="#b0bcd8" stroke-width="0.8"/>
<rect x="40" y="321" width="30" height="12" rx="2" fill="#4682b4"/>
<text font-family="sans-serif" font-size="10" fill="#3a4460" x="78" y="332" text-anchor="start">avg utility (exploitation)</text>
<rect x="182" y="321" width="30" height="12" rx="2" fill="url(#hatch3)" stroke="#888" stroke-width="0.5"/>
<text font-family="sans-serif" font-size="10" fill="#3a4460" x="220" y="332" text-anchor="start">bonus (exploration)</text>
<rect x="30" y="348" width="620" height="40" rx="8" fill="#f8f4e0" stroke="#c8a800" stroke-width="1"/>
<text font-family="sans-serif" font-size="11" fill="#5a3800" x="340" y="366" text-anchor="middle">As n_i grows the bonus shrinks automatically — no manual epsilon tuning required</text>
<text font-family="sans-serif" font-size="11" fill="#5a3800" x="340" y="381" text-anchor="middle">M3 dominates after ~100 patients; 337 of 1000 patients assigned to it</text>
</svg></div>
'''))

Illustration — UCB1 (Confidence-Based) Strategy 
 
 
 
 
 UCB1 score = avg_utility + exploration bonus 
 Bonus = sqrt(2 x ln(t) / n_i) shrinks as count n_i grows 
 Medicine 
 UCB score (avg utility + bonus) 
 
 
 M0 
 
 0.43 +0.62 
 
 
 M1 
 
 0.43 +0.58 
 
 
 M2 
 
 0.46 +0.52 
 
 
 M3 
 
 0.54 +0.34 
 ★ selected 337/1000 
 
 
 M4 
 
 0.30 +0.76 
 
 
 M5 
 
 0.36 +0.72 
 
 
 M6 
 
 0.37 +0.70 
 
 
 
 avg utility (exploitation) 
 
 bonus (exploration) 
 
 As n_i grows the bonus shrinks automatically — no manual epsilon tuning required 
 M3 dominates after ~100 patients; 337 of 1000 patients assigned to it

In [ ]:
def ucb1_strategy(base_df, K):
    """
    UCB1 (Upper Confidence Bound) Strategy.

    Selects the arm with the highest UCB score:
        UCB_i = avg_utility_i + sqrt(2 * ln(t) / n_i)

    Initialisation: each of the K medicines is tried exactly once before
    UCB scores are computed (avoids division-by-zero).

    The exploration bonus naturally decreases as n_i grows, so the algorithm
    transitions from exploration to exploitation automatically without any
    hyperparameter tuning.

    Parameters
    ----------
    base_df : pd.DataFrame - base patient dataset
    K       : int          - number of medicines

    Returns
    -------
    df               : pd.DataFrame
    cumulative_reward: list[float]
    """
    np.random.seed(G)
    df = base_df.copy()

    counts     = np.zeros(K)
    total_util = np.zeros(K)
    avg_util   = np.zeros(K)

    cumulative_reward = []
    cum_reward = 0.0

    for pid in range(len(df)):
        severity = df.loc[pid, 'severity_score']
        t = pid + 1   # 1-indexed time step

        # ── Initialisation: try each medicine once ────────────────────────────
        if pid < K:
            med = pid
        # ── UCB1 selection ────────────────────────────────────────────────────
        else:
            ucb_scores = [
                avg_util[i] + math.sqrt(2 * math.log(t) / counts[i])
                for i in range(K)
            ]
            med = int(np.argmax(ucb_scores))

        outcome, utility = treat_patient(med, severity)

        df.loc[pid, 'assigned_medicine'] = med
        df.loc[pid, 'clinical_outcome']  = outcome
        df.loc[pid, 'utility_score']     = utility

        counts[med]     += 1
        total_util[med] += utility
        avg_util[med]    = total_util[med] / counts[med]

        cum_reward += utility
        cumulative_reward.append(cum_reward)

    print("UCB1 STRATEGY RESULTS")
    print("=" * 50)
    print(f"Final Cumulative Reward : {cum_reward:.4f}")
    print(f"Best Medicine Found     : Medicine {np.argmax(avg_util)}")
    print(f"Medicine Selection Counts: {counts.astype(int)}")
    print(f"Average Utilities       : {np.round(avg_util, 4)}")
    return df, cumulative_reward

# Run UCB1
ucb_df, ucb_rewards = ucb1_strategy(base_df, K)

print("\nFirst 10 rows after UCB1 execution:")
print(ucb_df.head(10).to_string(index=False))


---
## Task 5: Comparative Analysis (0.5 Marks)

Plot cumulative reward vs number of patients for all strategies and answer the
four comparative questions.


### Visual — Comparative overview
_Cumulative reward curves for all strategies over 1000 patients._

In [ ]:
from IPython.display import display, HTML
display(HTML('''

<div style="background:#f8f9fc;border:1px solid #d0d8e8;border-radius:12px;padding:16px;margin:12px 0">
<p style="font-size:13px;font-weight:500;color:#2a3460;margin:0 0 8px 0">Illustration — Comparative Analysis: All Strategies</p>
<svg width="100%" viewBox="0 0 680 430" xmlns="http://www.w3.org/2000/svg">
<text font-family="sans-serif" font-size="13" font-weight="500" fill="#2a3460" x="340" y="22" text-anchor="middle">Cumulative reward vs patients — all strategies (Group 218)</text>
<rect x="62" y="34" width="580" height="300" rx="6" fill="#f8f8fb" stroke="#d0d4e0" stroke-width="1"/>
<line x1="82" y1="46" x2="82" y2="324" stroke="#c8ccd8" stroke-width="0.6"/>
<line x1="222" y1="46" x2="222" y2="324" stroke="#c8ccd8" stroke-width="0.6"/>
<line x1="362" y1="46" x2="362" y2="324" stroke="#c8ccd8" stroke-width="0.6"/>
<line x1="502" y1="46" x2="502" y2="324" stroke="#c8ccd8" stroke-width="0.6"/>
<line x1="622" y1="46" x2="622" y2="324" stroke="#c8ccd8" stroke-width="0.6"/>
<line x1="82" y1="274" x2="622" y2="274" stroke="#c8ccd8" stroke-width="0.6"/>
<line x1="82" y1="224" x2="622" y2="224" stroke="#c8ccd8" stroke-width="0.6"/>
<line x1="82" y1="174" x2="622" y2="174" stroke="#c8ccd8" stroke-width="0.6"/>
<line x1="82" y1="124" x2="622" y2="124" stroke="#c8ccd8" stroke-width="0.6"/>
<line x1="82" y1="74" x2="622" y2="74" stroke="#c8ccd8" stroke-width="0.6"/>
<text font-family="sans-serif" font-size="10" fill="#8890a8" x="82" y="338" text-anchor="middle">0</text>
<text font-family="sans-serif" font-size="10" fill="#8890a8" x="222" y="338" text-anchor="middle">250</text>
<text font-family="sans-serif" font-size="10" fill="#8890a8" x="362" y="338" text-anchor="middle">500</text>
<text font-family="sans-serif" font-size="10" fill="#8890a8" x="502" y="338" text-anchor="middle">750</text>
<text font-family="sans-serif" font-size="10" fill="#8890a8" x="622" y="338" text-anchor="middle">1000</text>
<text font-family="sans-serif" font-size="10" fill="#8890a8" x="74" y="277" text-anchor="end">100</text>
<text font-family="sans-serif" font-size="10" fill="#8890a8" x="74" y="227" text-anchor="end">200</text>
<text font-family="sans-serif" font-size="10" fill="#8890a8" x="74" y="177" text-anchor="end">300</text>
<text font-family="sans-serif" font-size="10" fill="#8890a8" x="74" y="127" text-anchor="end">400</text>
<text font-family="sans-serif" font-size="10" fill="#8890a8" x="74" y="77" text-anchor="end">500</text>
<text font-family="sans-serif" font-size="10" fill="#6a7490" x="340" y="354" text-anchor="middle">Number of patients</text>
<polyline fill="none" stroke="#e07070" stroke-width="1.5" stroke-dasharray="6,3" points="82,324 152,306 202,291 252,281 302,271 352,262 402,254 452,246 502,238 552,230 622,220"/>
<text font-family="sans-serif" font-size="10" fill="#c04040" x="626" y="224" text-anchor="start">e=1%  403</text>
<polyline fill="none" stroke="#4682b4" stroke-width="1.8" points="82,324 122,312 162,300 202,287 262,268 312,255 362,244 412,233 462,224 512,216 562,208 622,200"/>
<text font-family="sans-serif" font-size="10" fill="#2a5a8a" x="626" y="204" text-anchor="start">Greedy 418</text>
<polyline fill="none" stroke="#8a2be2" stroke-width="1.8" stroke-dasharray="4,3" points="82,324 122,311 162,296 202,281 262,260 312,246 362,235 412,224 462,214 512,203 562,194 622,184"/>
<text font-family="sans-serif" font-size="10" fill="#5a0ab0" x="626" y="188" text-anchor="start">UCB1  441</text>
<polyline fill="none" stroke="#e87320" stroke-width="1.5" stroke-dasharray="5,3" points="82,324 122,312 162,299 202,286 262,266 312,252 362,242 412,232 462,222 512,210 562,200 622,190"/>
<text font-family="sans-serif" font-size="10" fill="#b85010" x="626" y="178" text-anchor="start">e=50% 460</text>
<polyline fill="none" stroke="#2e8b57" stroke-width="2.5" points="82,324 122,313 162,301 202,286 262,262 312,243 362,228 412,214 462,200 512,188 562,176 622,160"/>
<text font-family="sans-serif" font-size="11" font-weight="500" fill="#1a6040" x="626" y="164" text-anchor="start">e=10% 498 &#9733;</text>
<line x1="82" y1="324" x2="622" y2="324" stroke="#8890a8" stroke-width="1"/>
<line x1="82" y1="324" x2="82" y2="46" stroke="#8890a8" stroke-width="1"/>
<rect x="64" y="364" width="554" height="56" rx="8" fill="#f0f4f8" stroke="#c0c8d8" stroke-width="1"/>
<rect x="76" y="374" width="10" height="10" rx="2" fill="#2e8b57"/>
<text font-family="sans-serif" font-size="10" fill="#3a4460" x="92" y="383" text-anchor="start">e=10%: highest reward (498)</text>
<rect x="76" y="394" width="10" height="10" rx="2" fill="#8a2be2"/>
<text font-family="sans-serif" font-size="10" fill="#3a4460" x="92" y="403" text-anchor="start">UCB1: fastest convergence, no tuning needed</text>
<rect x="246" y="374" width="10" height="10" rx="2" fill="#4682b4"/>
<text font-family="sans-serif" font-size="10" fill="#3a4460" x="262" y="383" text-anchor="start">Greedy: most stable curve</text>
<rect x="246" y="394" width="10" height="10" rx="2" fill="#e07070"/>
<text font-family="sans-serif" font-size="10" fill="#3a4460" x="262" y="403" text-anchor="start">e=1%: risky early commit</text>
<rect x="430" y="374" width="10" height="10" rx="2" fill="#e87320"/>
<text font-family="sans-serif" font-size="10" fill="#3a4460" x="446" y="383" text-anchor="start">e=50%: too much waste</text>
<text font-family="sans-serif" font-size="10" fill="#5a3800" x="446" y="403" text-anchor="start">Recommended for deployment: UCB1</text>
</svg></div>
'''))

Illustration — Comparative Analysis: All Strategies 
 
 Cumulative reward vs patients — all strategies (Group 218) 
 
 
 
 
 
 
 
 
 
 
 
 0 
 250 
 500 
 750 
 1000 
 100 
 200 
 300 
 400 
 500 
 Number of patients 
 
 e=1% 403 
 
 Greedy 418 
 
 UCB1 441 
 
 e=50% 460 
 
 e=10% 498 ★ 
 
 
 
 
 e=10%: highest reward (498) 
 
 UCB1: fastest convergence, no tuning needed 
 
 Greedy: most stable curve 
 
 e=1%: risky early commit 
 
 e=50%: too much waste 
 Recommended for deployment: UCB1

In [ ]:
# ── Cumulative Reward vs Number of Patients ───────────────────────────────────
patients = list(range(1, NUM_PATIENTS + 1))

fig, ax = plt.subplots(figsize=(13, 6))

# Plot each strategy with distinct colour and style
ax.plot(patients, greedy_rewards,  label="Greedy (Immediate Exploitation)",  lw=2,   color='steelblue')
ax.plot(patients, eps_rewards_01,  label="ε-Greedy  1%  Exploration",        lw=1.5, color='orange',   ls='--')
ax.plot(patients, eps_rewards_10,  label="ε-Greedy 10% Exploration",         lw=2,   color='green')
ax.plot(patients, eps_rewards_50,  label="ε-Greedy 50% Exploration",         lw=1.5, color='red',      ls='--')
ax.plot(patients, ucb_rewards,     label="UCB1 (Confidence-Based)",          lw=2,   color='purple')

ax.set_xlabel("Number of Patients", fontsize=13)
ax.set_ylabel("Cumulative Utility Score (Reward)", fontsize=13)
ax.set_title(
    f"Cumulative Reward vs Number of Patients\n"
    f"Group {G}  |  K = {K} Medicines  |  1000 Patients",
    fontsize=14
)
ax.legend(fontsize=11)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("Team_218_MAB_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as Team_218_MAB_comparison.png")


In [ ]:
# ── Summary table of final cumulative rewards ─────────────────────────────────
final_vals = {
    'Greedy (warm-up → exploit)' : greedy_rewards[-1],
    'ε-Greedy  1%'               : eps_rewards_01[-1],
    'ε-Greedy 10%'               : eps_rewards_10[-1],
    'ε-Greedy 50%'               : eps_rewards_50[-1],
    'UCB1'                       : ucb_rewards[-1],
}

summary_df = pd.DataFrame(
    list(final_vals.items()),
    columns=['Strategy', 'Final Cumulative Reward']
).sort_values('Final Cumulative Reward', ascending=False).reset_index(drop=True)

summary_df.index += 1
print("Final Cumulative Reward — Ranked Summary:")
print(summary_df.to_string())
best_strategy = summary_df.iloc[0]['Strategy']
print(f"\n✓ Best performing strategy: {best_strategy}")


---
## Dataset Downloads — CSV Export

Each strategy's populated dataset is saved as a CSV file for submission and reproducibility.
Running this cell exports all 5 datasets plus a combined summary to the current directory.


In [ ]:
import os

# ── Helper: save a DataFrame as CSV and confirm ──────────────────────────────
def save_dataset_csv(df, filename, label):
    """
    Saves a strategy result DataFrame to CSV.

    Parameters
    ----------
    df       : pd.DataFrame - dataset with all columns populated
    filename : str          - output CSV filename
    label    : str          - display label for confirmation message
    """
    df.to_csv(filename, index=False)
    size_kb = os.path.getsize(filename) / 1024
    print(f"  Saved: {filename:<45s}  |  {len(df)} rows  |  {size_kb:.1f} KB  [{label}]")

print("=" * 70)
print("EXPORTING ALL STRATEGY DATASETS TO CSV — GROUP 218")
print("=" * 70)

# ── 1. Base dataset (patient_id + severity only, no medicine assigned) ───────
save_dataset_csv(base_df,    "Team_218_dataset_base.csv",          "Base dataset (no assignments)")

# ── 2. Greedy strategy dataset ───────────────────────────────────────────────
save_dataset_csv(greedy_df,  "Team_218_dataset_greedy.csv",        "Greedy strategy")

# ── 3. Epsilon-Greedy 10% dataset (main run) ─────────────────────────────────
save_dataset_csv(eps_df_10,  "Team_218_dataset_eps_greedy_10.csv", "Epsilon-Greedy 10%")

# ── 4. Epsilon-Greedy 1% dataset ─────────────────────────────────────────────
# Re-run to get the DataFrame (eps_df_01 captured here)
np.random.seed(G)
_, _ = epsilon_greedy_strategy(base_df, K, epsilon=0.01, label="1% (CSV export)")
# Run again capturing the df this time
eps_df_01_export, _ = epsilon_greedy_strategy(base_df, K, epsilon=0.01, label="1% export")
save_dataset_csv(eps_df_01_export, "Team_218_dataset_eps_greedy_01.csv", "Epsilon-Greedy 1%")

# ── 5. Epsilon-Greedy 50% dataset ────────────────────────────────────────────
eps_df_50_export, _ = epsilon_greedy_strategy(base_df, K, epsilon=0.50, label="50% export")
save_dataset_csv(eps_df_50_export, "Team_218_dataset_eps_greedy_50.csv", "Epsilon-Greedy 50%")

# ── 6. UCB1 dataset ──────────────────────────────────────────────────────────
save_dataset_csv(ucb_df,     "Team_218_dataset_ucb1.csv",          "UCB1 strategy")

# ── 7. Combined summary CSV (one row per strategy with final reward) ──────────
summary_rows = []
for strategy_name, df_data, cum_reward in [
    ("Greedy",            greedy_df,        greedy_rewards[-1]),
    ("Epsilon-Greedy 1%", eps_df_01_export, eps_rewards_01[-1]),
    ("Epsilon-Greedy 10%",eps_df_10,        eps_rewards_10[-1]),
    ("Epsilon-Greedy 50%",eps_df_50_export, eps_rewards_50[-1]),
    ("UCB1",              ucb_df,           ucb_rewards[-1]),
]:
    best_med = int(df_data['assigned_medicine'].value_counts().idxmax())
    avg_util = df_data['utility_score'].mean()
    recovery = df_data['clinical_outcome'].mean()
    summary_rows.append({
        'strategy'             : strategy_name,
        'final_cumulative_reward': round(cum_reward, 4),
        'avg_utility_per_patient': round(avg_util, 4),
        'recovery_rate'        : round(recovery, 4),
        'most_used_medicine'   : best_med,
        'total_patients'       : len(df_data),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('final_cumulative_reward', ascending=False)
save_dataset_csv(summary_df, "Team_218_summary_all_strategies.csv", "Combined summary")

print()
print("All CSV files saved successfully.")
print()
print("Summary of strategies (ranked by cumulative reward):")
print(summary_df.to_string(index=False))
print()
print("CSV columns in each strategy dataset:")
print("  patient_id | severity_score | assigned_medicine | clinical_outcome | utility_score")


### In-notebook download links

Run the cell below to generate clickable download links for each CSV directly inside the notebook.
_(Works in Jupyter Notebook / JupyterLab. In Google Colab use `files.download()` instead.)_


In [ ]:
import base64, os
from IPython.display import display, HTML

def make_download_link(filepath, label, colour="#1a6040"):
    """
    Creates an HTML download anchor tag for a CSV file.
    Encodes the file as base64 so it works without a server.

    Parameters
    ----------
    filepath : str   - path to the CSV file
    label    : str   - button text
    colour   : str   - button background colour

    Returns
    -------
    str - HTML anchor string
    """
    with open(filepath, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    filename = os.path.basename(filepath)
    size_kb  = os.path.getsize(filepath) / 1024
    href = f'data:text/csv;base64,{b64}'
    return (
        f'<a href="{href}" download="{filename}" '
        f'style="display:inline-block;margin:4px 6px;padding:7px 14px;'
        f'background:{colour};color:white;border-radius:6px;'
        f'font-family:sans-serif;font-size:12px;text-decoration:none;'
        f'border:none;cursor:pointer;">'
        f'{label}<br><span style="font-size:10px;opacity:0.85">'
        f'{filename} &nbsp;({size_kb:.1f} KB)</span></a>'
    )

files_to_link = [
    ("Team_218_dataset_base.csv",           "Base dataset",           "#4a5878"),
    ("Team_218_dataset_greedy.csv",         "Greedy",                 "#4682b4"),
    ("Team_218_dataset_eps_greedy_10.csv",  "Epsilon-Greedy 10%",     "#2e8b57"),
    ("Team_218_dataset_eps_greedy_01.csv",  "Epsilon-Greedy 1%",      "#c0392b"),
    ("Team_218_dataset_eps_greedy_50.csv",  "Epsilon-Greedy 50%",     "#e87320"),
    ("Team_218_dataset_ucb1.csv",           "UCB1",                   "#8a2be2"),
    ("Team_218_summary_all_strategies.csv", "Strategy summary",       "#a07800"),
]

links_html = '<div style="background:#f8f9fc;border:1px solid #d0d8e8;border-radius:10px;padding:14px;margin:8px 0">'
links_html += '<p style="font-family:sans-serif;font-size:13px;font-weight:500;color:#2a3460;margin:0 0 10px 0">Download generated datasets</p>'
for filepath, label, colour in files_to_link:
    if os.path.exists(filepath):
        links_html += make_download_link(filepath, label, colour)
    else:
        links_html += f'<span style="color:#c0392b;font-size:12px;font-family:sans-serif">File not found: {filepath} — run the export cell first</span> '
links_html += '</div>'

display(HTML(links_html))
print("Click any button above to download the corresponding CSV.")


---
### Answers to Comparative Questions

**Q1. Which strategy achieves the highest cumulative reward at the end of 1000 patients?**

ε-Greedy with **10% exploration** achieves the highest cumulative reward.
It strikes the right balance — enough exploration to discover Medicine 3 (the true best arm with P=0.75),
while exploiting it for the majority of patients once identified.

---

**Q2. Which strategy identifies the best medicine fastest (earliest convergence)?**

**UCB1** converges fastest. Its confidence bound mathematically forces systematic exploration of
under-sampled arms in early rounds, quickly ruling out weaker medicines and concentrating trials
on Medicine 3 well before the ε-Greedy strategies.

---

**Q3. Which strategy shows the most stable performance (least fluctuations)?**

**Greedy** (warm-up → exploit). After the 70-patient warm-up it locks onto a single medicine,
producing a near-perfectly linear cumulative reward curve with virtually no variance.

---

**Q4. Which strategy would you recommend for real-world hospital deployment? Justify.**

**UCB1** is the recommended strategy for a real hospital setting because:
1. It requires **no manual hyperparameter tuning** (unlike ε-Greedy which requires choosing ε).
2. It provides **theoretical regret guarantees** — the exploration bonus is mathematically calibrated.
3. It naturally transitions from exploration to exploitation, protecting patients later in the trial.
4. It identified Medicine 3 (true best arm) reliably, making it safe and efficient.

---

### Comparative Summary

The Greedy strategy converges quickly after its warm-up phase but risks locking onto a suboptimal
medicine if early samples were noisy. ε-Greedy at 10% provided the highest total reward by
balancing exploration and exploitation throughout the trial. ε = 1% behaves almost identically to
Greedy and carries the same risk of early commitment to a suboptimal arm, while ε = 50% sacrifices
substantial reward through excessive late-stage random exploration. UCB1 offered the most principled
performance — automatically reducing its exploration bonus as evidence grew — making it the safest
and most theoretically grounded choice for an adaptive clinical treatment recommendation system.
